# Phase 2: Batch Data Preprocessing & Event Grouping

## Overview
This notebook processes all 27 datasets from Phase 1 parsing output into grouped event CSVs for Phase 3 feature engineering.

## Dataset Split
**Training Datasets (22):**
- PE: 01-PE through 12-PE (12 datasets)
- APT: 01-APT17, 03-APT21, 04-APT28, 05-APT29, 06-APT30, 07-APT37, 08-APT38, 10-DarkHotel663, 11-DarkHotelbbd, 14-Winnti43b (10 datasets)

**Validation Datasets (5):**
- APT: 02-APT19, 09-APT40, 12-Kimsuky, 13-Winnti731 (4 datasets)
- LoneWolf: LoneWolf (1 dataset)

## Output Structure
- Individual: `grouped_events_[dataset].csv` for each dataset
- Combined Training: `grouped_events_training.csv` (all 22 training datasets merged)
- Validation datasets remain separate for independent evaluation

## Processing Steps
1. Load MFT, LogFile, UsnJrnl from Phase 1
2. Join events to MFT entries (TargetFRN/FRN -> EntryNumber)
3. Normalize event schema
4. Combine and sort events per file
5. Export grouped events CSV


In [11]:
# [Cell 2] Imports and Global Configuration

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# DIRECTORY CONFIGURATION
# =============================================================================

# Phase 1 input directories
PHASE1_BASE = Path("/Users/soni/Github/Digital-Detectives_Thesis/data/Phase 1: Raw Data Parsing")
PE_INPUT_DIR = PHASE1_BASE / "PE"
APT_INPUT_DIR = PHASE1_BASE / "APT"
LONEWOLF_INPUT_DIR = PHASE1_BASE / "LoneWolf" / "LoneWolf"

# Phase 2 output directory
OUTPUT_DIR = Path("/Users/soni/Github/Digital-Detectives_Thesis/data/Phase 2: Data Preprocessing & Event Grouping")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =============================================================================
# DATASET DEFINITIONS
# =============================================================================

# Training datasets (22 total)
TRAINING_PE = [f"{i:02d}-PE" for i in range(1, 13)]  # 01-PE to 12-PE

TRAINING_APT = [
    "01-APT17", "03-APT21", "04-APT28", "05-APT29", "06-APT30",
    "07-APT37", "08-APT38", "10-DarkHotel663", "11-DarkHotelbbd", "14-Winnti43b"
]

# Validation datasets (5 total)
VALIDATION_APT = ["02-APT19", "09-APT40", "12-Kimsuky", "13-Winnti731"]
VALIDATION_LONEWOLF = ["LoneWolf"]

# Combined lists
ALL_TRAINING = TRAINING_PE + TRAINING_APT
ALL_VALIDATION = VALIDATION_APT + VALIDATION_LONEWOLF

print(f"Training datasets: {len(ALL_TRAINING)}")
print(f"  PE: {len(TRAINING_PE)}")
print(f"  APT: {len(TRAINING_APT)}")
print(f"\nValidation datasets: {len(ALL_VALIDATION)}")
print(f"  APT: {len(VALIDATION_APT)}")
print(f"  LoneWolf: {len(VALIDATION_LONEWOLF)}")
print(f"\nTotal datasets: {len(ALL_TRAINING) + len(ALL_VALIDATION)}")


Training datasets: 22
  PE: 12
  APT: 10

Validation datasets: 5
  APT: 4
  LoneWolf: 1

Total datasets: 27


## Core Processing Functions

All preprocessing functions are defined below. Run this cell once before processing any datasets.

### Functions Overview
1. `get_input_paths()` - Resolve input file paths for a dataset
2. `load_phase1_data()` - Load MFT, LogFile, UsnJrnl CSVs
3. `preprocess_mft()` - Create MFT reference table
4. `process_logfile_events()` - Process and join LogFile events
5. `process_usnjrnl_events()` - Process and join UsnJrnl events
6. `normalize_logfile_schema()` - Normalize LogFile to unified schema
7. `normalize_usnjrnl_schema()` - Normalize UsnJrnl to unified schema
8. `combine_and_sort_events()` - Merge and order event streams
9. `process_dataset()` - Main function to process a single dataset


In [12]:
# [Cell 4] Path Resolution Function

def get_input_paths(dataset_name, category):
    """
    Resolve input file paths for a dataset.
    
    Parameters:
        dataset_name: Name of the dataset (e.g., "01-PE", "01-APT17", "LoneWolf")
        category: One of "PE", "APT", or "LoneWolf"
    
    Returns:
        dict with keys: logfile, mft, usnjrnl, input_dir
    """
    if category == "PE":
        input_dir = PE_INPUT_DIR / dataset_name
        prefix = dataset_name
    elif category == "APT":
        input_dir = APT_INPUT_DIR / dataset_name
        prefix = dataset_name
    elif category == "LoneWolf":
        input_dir = LONEWOLF_INPUT_DIR
        prefix = "LoneWolf"
    else:
        raise ValueError(f"Unknown category: {category}")
    
    paths = {
        'input_dir': input_dir,
        'logfile': input_dir / f"{prefix}-LogFile.csv",
        'mft': input_dir / f"{prefix}-MFT.csv",
        'usnjrnl': input_dir / f"{prefix}-UsnJrnl.csv"
    }
    
    return paths

def validate_input_paths(paths):
    """Check if all input files exist."""
    missing = []
    for key, path in paths.items():
        if key != 'input_dir' and not path.exists():
            missing.append(str(path))
    return missing

print("Path resolution functions defined.")


Path resolution functions defined.


In [13]:
# [Cell 5] Data Loading Function

def load_phase1_data(paths):
    """
    Load MFT, LogFile, and UsnJrnl CSVs from Phase 1 output.
    
    Parameters:
        paths: dict from get_input_paths()
    
    Returns:
        dict with keys: mft, logfile, usnjrnl (DataFrames)
    """
    print("  Loading MFT...")
    df_mft = pd.read_csv(paths['mft'], low_memory=False)
    print(f"    MFT records: {len(df_mft):,}")
    
    print("  Loading LogFile...")
    df_logfile = pd.read_csv(paths['logfile'], low_memory=False)
    print(f"    LogFile records: {len(df_logfile):,}")
    
    print("  Loading UsnJrnl...")
    df_usnjrnl = pd.read_csv(paths['usnjrnl'], low_memory=False)
    print(f"    UsnJrnl records: {len(df_usnjrnl):,}")
    
    return {
        'mft': df_mft,
        'logfile': df_logfile,
        'usnjrnl': df_usnjrnl
    }


print("Data loading function defined.")


Data loading function defined.


In [14]:
# [Cell 6] MFT Preprocessing Function

def preprocess_mft(df_mft):
    """
    Create MFT reference table for joining with LogFile and UsnJrnl.
    
    Parameters:
        df_mft: Raw MFT DataFrame
    
    Returns:
        df_mft_ref: Preprocessed MFT reference table with FileFRN as key
    """
    mft_columns = [
        'EntryNumber', 'FileName', 'FilePath', 'IsActive', 'LSN', 'ParentFRN',
        '$SI-C', '$SI-M', '$SI-E', '$SI-A',
        '$FN-C', '$FN-M', '$FN-E', '$FN-A'
    ]
    
    # Select columns that exist
    available_cols = [c for c in mft_columns if c in df_mft.columns]
    df_mft_ref = df_mft[available_cols].copy()
    
    # Rename EntryNumber to FileFRN
    df_mft_ref = df_mft_ref.rename(columns={'EntryNumber': 'FileFRN'})
    
    # Handle missing values
    if 'FileName' in df_mft_ref.columns:
        df_mft_ref['FileName'] = df_mft_ref['FileName'].fillna('')
    if 'FilePath' in df_mft_ref.columns:
        df_mft_ref['FilePath'] = df_mft_ref['FilePath'].fillna('')
    
    return df_mft_ref

print("MFT preprocessing function defined.")


MFT preprocessing function defined.


In [15]:
# [Cell 7] LogFile Processing Function

def process_logfile_events(df_logfile, df_mft_ref):
    """
    Process LogFile events and join with MFT reference.
    
    Parameters:
        df_logfile: Raw LogFile DataFrame
        df_mft_ref: Preprocessed MFT reference table
    
    Returns:
        df_logfile_joined: LogFile events with MFT metadata
    """
    # Clean TargetFRN
    df_logfile = df_logfile.copy()
    df_logfile['TargetFRN_clean'] = pd.to_numeric(df_logfile['TargetFRN'], errors='coerce')
    df_logfile['TargetFRN_clean'] = df_logfile['TargetFRN_clean'].fillna(-1).astype(int)
    
    # Filter valid records
    df_logfile_valid = df_logfile[df_logfile['TargetFRN_clean'] >= 0].copy()
    
    # Join with MFT
    df_logfile_joined = df_logfile_valid.merge(
        df_mft_ref[['FileFRN', 'FileName', 'FilePath', 'IsActive']],
        left_on='TargetFRN_clean',
        right_on='FileFRN',
        how='left'
    )
    
    # Add event source
    df_logfile_joined['EventSource'] = 'LogFile'
    
    # Set EventTimestamp from Redo_$SI-E
    df_logfile_joined['EventTimestamp'] = df_logfile_joined.get('Redo_$SI-E', np.nan)
    
    return df_logfile_joined


print("LogFile processing function defined.")


LogFile processing function defined.


In [16]:
# [Cell 8] UsnJrnl Processing Function

def process_usnjrnl_events(df_usnjrnl, df_mft_ref):
    """
    Process UsnJrnl events and join with MFT reference.
    
    Parameters:
        df_usnjrnl: Raw UsnJrnl DataFrame
        df_mft_ref: Preprocessed MFT reference table
    
    Returns:
        df_usnjrnl_joined: UsnJrnl events with MFT metadata
    """
    # Clean FRN
    df_usnjrnl = df_usnjrnl.copy()
    df_usnjrnl['FRN_clean'] = pd.to_numeric(df_usnjrnl['FRN'], errors='coerce')
    df_usnjrnl['FRN_clean'] = df_usnjrnl['FRN_clean'].fillna(-1).astype(int)
    
    # Filter valid records
    df_usnjrnl_valid = df_usnjrnl[df_usnjrnl['FRN_clean'] >= 0].copy()
    
    # Join with MFT
    df_usnjrnl_joined = df_usnjrnl_valid.merge(
        df_mft_ref[['FileFRN', 'FileName', 'FilePath', 'IsActive']],
        left_on='FRN_clean',
        right_on='FileFRN',
        how='left',
        suffixes=('_usn', '_mft')
    )
    
    # Use UsnJrnl filename, fallback to MFT
    if 'FileName_usn' in df_usnjrnl_joined.columns:
        df_usnjrnl_joined['FileName'] = df_usnjrnl_joined['FileName_usn'].fillna(
            df_usnjrnl_joined.get('FileName_mft', '')
        )
        df_usnjrnl_joined = df_usnjrnl_joined.drop(
            columns=[c for c in ['FileName_usn', 'FileName_mft'] if c in df_usnjrnl_joined.columns]
        )
    
    # Add event source
    df_usnjrnl_joined['EventSource'] = 'UsnJrnl'
    
    # Set EventTimestamp from Timestamp
    df_usnjrnl_joined['EventTimestamp'] = df_usnjrnl_joined.get('Timestamp', np.nan)
    
    return df_usnjrnl_joined


print("UsnJrnl processing function defined.")


UsnJrnl processing function defined.


In [17]:
# [Cell 9] Schema Normalization Functions

# Define the final unified column order
FINAL_COLUMNS = [
    # Identifiers
    'dataID', 'FileFRN', 'FileName', 'FilePath', 'EventSource', 'EventTimestamp',
    # Sequence numbers
    'LSN', 'USN',
    # LogFile fields
    'RedoOP', 'UndoOP', 'RedoOPName', 'UndoOPName',
    'RecordOffset', 'AttributeOffset', 'TargetVCN',
    'IsTimestampChange',
    # LogFile timestamps
    'Undo_$SI-C', 'Undo_$SI-M', 'Undo_$SI-E', 'Undo_$SI-A',
    'Redo_$SI-C', 'Redo_$SI-M', 'Redo_$SI-E', 'Redo_$SI-A',
    # UsnJrnl fields
    'ReasonCode', 'ReasonFlags',
    'HasBasicInfoChange', 'HasClose', 'HasFileCreate'
]


def normalize_logfile_schema(df_logfile_joined, data_id):
    """
    Normalize LogFile events to unified schema.
    
    Parameters:
        df_logfile_joined: Processed LogFile DataFrame
        data_id: Dataset identifier string
    
    Returns:
        DataFrame with unified schema
    """
    df = df_logfile_joined.copy()
    
    # Add dataID
    df['dataID'] = data_id
    
    # Add placeholder columns for UsnJrnl fields
    df['USN'] = np.nan
    df['ReasonCode'] = np.nan
    df['ReasonFlags'] = ''
    df['HasBasicInfoChange'] = False
    df['HasClose'] = False
    df['HasFileCreate'] = False
    
    # Ensure all columns exist
    for col in FINAL_COLUMNS:
        if col not in df.columns:
            df[col] = np.nan if col not in ['ReasonFlags', 'RedoOPName', 'UndoOPName'] else ''
    
    return df[FINAL_COLUMNS]


def normalize_usnjrnl_schema(df_usnjrnl_joined, data_id):
    """
    Normalize UsnJrnl events to unified schema.
    
    Parameters:
        df_usnjrnl_joined: Processed UsnJrnl DataFrame
        data_id: Dataset identifier string
    
    Returns:
        DataFrame with unified schema
    """
    df = df_usnjrnl_joined.copy()
    
    # Add dataID
    df['dataID'] = data_id
    
    # Add placeholder columns for LogFile fields
    df['LSN'] = np.nan
    df['RedoOP'] = np.nan
    df['UndoOP'] = np.nan
    df['RedoOPName'] = ''
    df['UndoOPName'] = ''
    df['RecordOffset'] = np.nan
    df['AttributeOffset'] = np.nan
    df['TargetVCN'] = np.nan
    df['IsTimestampChange'] = False
    
    # Undo/Redo timestamps not applicable
    for ts_col in ['Undo_$SI-C', 'Undo_$SI-M', 'Undo_$SI-E', 'Undo_$SI-A',
                   'Redo_$SI-C', 'Redo_$SI-M', 'Redo_$SI-E', 'Redo_$SI-A']:
        df[ts_col] = ''
    
    # Ensure all columns exist
    for col in FINAL_COLUMNS:
        if col not in df.columns:
            df[col] = np.nan if col not in ['ReasonFlags', 'RedoOPName', 'UndoOPName'] else ''
    
    return df[FINAL_COLUMNS]


print("Schema normalization functions defined.")
print(f"Unified schema has {len(FINAL_COLUMNS)} columns.")


Schema normalization functions defined.
Unified schema has 29 columns.


In [18]:
# [Cell 10] Combine and Sort Function

def combine_and_sort_events(df_logfile_norm, df_usnjrnl_norm):
    """
    Combine LogFile and UsnJrnl events and sort by file and timestamp.
    
    Parameters:
        df_logfile_norm: Normalized LogFile DataFrame
        df_usnjrnl_norm: Normalized UsnJrnl DataFrame
    
    Returns:
        df_grouped: Combined and sorted DataFrame
    """
    # Combine event streams
    df_combined = pd.concat([df_logfile_norm, df_usnjrnl_norm], ignore_index=True)
    
    # Convert EventTimestamp
    df_combined['EventTimestamp'] = pd.to_datetime(df_combined['EventTimestamp'], errors='coerce')
    
    # Create sort keys
    sentinel_date = pd.Timestamp('1970-01-01')
    df_combined['_sort_ts'] = df_combined['EventTimestamp'].fillna(sentinel_date)
    df_combined['_sort_seq'] = df_combined['LSN'].fillna(0) + df_combined['USN'].fillna(0)
    
    # Sort by FileFRN, timestamp, sequence
    df_grouped = df_combined.sort_values(
        by=['FileFRN', '_sort_ts', '_sort_seq'],
        ascending=[True, True, True]
    ).reset_index(drop=True)
    
    # Remove temporary columns
    df_grouped = df_grouped.drop(columns=['_sort_ts', '_sort_seq'])
    
    return df_grouped


print("Combine and sort function defined.")


Combine and sort function defined.


In [19]:
# [Cell 11] Main Processing Function

def process_dataset(dataset_name, category, output_dir=OUTPUT_DIR):
    """
    Process a single dataset through Phase 2 pipeline.
    
    Parameters:
        dataset_name: Name of the dataset (e.g., "01-PE", "01-APT17")
        category: One of "PE", "APT", or "LoneWolf"
        output_dir: Output directory path
    
    Returns:
        dict with processing statistics
    """
    print(f"\n{'='*60}")
    print(f"PROCESSING: {dataset_name} ({category})")
    print(f"{'='*60}")
    
    # Get input paths
    paths = get_input_paths(dataset_name, category)
    
    # Validate inputs
    missing = validate_input_paths(paths)
    if missing:
        print(f"  ERROR: Missing files: {missing}")
        return None
    
    # Load data
    data = load_phase1_data(paths)
    
    # Preprocess MFT
    print("  Preprocessing MFT reference table...")
    df_mft_ref = preprocess_mft(data['mft'])
    
    # Process LogFile
    print("  Processing LogFile events...")
    df_logfile_joined = process_logfile_events(data['logfile'], df_mft_ref)
    
    # Process UsnJrnl
    print("  Processing UsnJrnl events...")
    df_usnjrnl_joined = process_usnjrnl_events(data['usnjrnl'], df_mft_ref)
    
    # Normalize schemas
    print("  Normalizing event schemas...")
    df_logfile_norm = normalize_logfile_schema(df_logfile_joined, dataset_name)
    df_usnjrnl_norm = normalize_usnjrnl_schema(df_usnjrnl_joined, dataset_name)
    
    # Combine and sort
    print("  Combining and sorting events...")
    df_grouped = combine_and_sort_events(df_logfile_norm, df_usnjrnl_norm)
    
    # Export
    output_path = output_dir / f"grouped_events_{dataset_name}.csv"
    df_grouped.to_csv(output_path, index=False, encoding='utf-8')
    output_size_mb = output_path.stat().st_size / (1024 * 1024)
    
    # Compute statistics
    stats = {
        'dataset': dataset_name,
        'category': category,
        'mft_records': len(data['mft']),
        'logfile_records': len(data['logfile']),
        'usnjrnl_records': len(data['usnjrnl']),
        'logfile_events': len(df_logfile_norm),
        'usnjrnl_events': len(df_usnjrnl_norm),
        'total_events': len(df_grouped),
        'unique_files': df_grouped['FileFRN'].nunique(),
        'timestamp_changes': df_grouped['IsTimestampChange'].sum() if 'IsTimestampChange' in df_grouped.columns else 0,
        'output_size_mb': output_size_mb,
        'output_path': str(output_path)
    }
    
    # Print summary
    print(f"\n  --- Summary ---")
    print(f"  Total events: {stats['total_events']:,}")
    print(f"  Unique files: {stats['unique_files']:,}")
    print(f"  Timestamp changes: {stats['timestamp_changes']:,}")
    print(f"  Output: {output_path.name} ({output_size_mb:.2f} MB)")
    
    return stats


print("Main processing function defined.")
print("\nReady to process datasets. Run individual dataset cells below.")


Main processing function defined.

Ready to process datasets. Run individual dataset cells below.


## Training Datasets: PE (01-PE to 12-PE)

Run each cell individually to process one dataset at a time.
All outputs saved to Phase 2 output directory.


In [20]:
# [Cell 13] Process 01-PE
stats_01PE = process_dataset("01-PE", "PE")



PROCESSING: 01-PE (PE)
  Loading MFT...
    MFT records: 519,113
  Loading LogFile...
    LogFile records: 417,267
  Loading UsnJrnl...
    UsnJrnl records: 316,817
  Preprocessing MFT reference table...
  Processing LogFile events...
  Processing UsnJrnl events...
  Normalizing event schemas...
  Combining and sorting events...

  --- Summary ---
  Total events: 505,582
  Unique files: 37,375
  Timestamp changes: 32,435
  Output: grouped_events_01-PE.csv (122.67 MB)


In [21]:
# [Cell 14] Process 02-PE
stats_02PE = process_dataset("02-PE", "PE")



PROCESSING: 02-PE (PE)
  Loading MFT...
    MFT records: 685,470
  Loading LogFile...
    LogFile records: 294,846
  Loading UsnJrnl...
    UsnJrnl records: 247,386
  Preprocessing MFT reference table...
  Processing LogFile events...
  Processing UsnJrnl events...
  Normalizing event schemas...
  Combining and sorting events...

  --- Summary ---
  Total events: 363,619
  Unique files: 105,251
  Timestamp changes: 14,978
  Output: grouped_events_02-PE.csv (95.41 MB)


In [22]:
# [Cell 15] Process 03-PE
stats_03PE = process_dataset("03-PE", "PE")



PROCESSING: 03-PE (PE)
  Loading MFT...
    MFT records: 685,469
  Loading LogFile...
    LogFile records: 302,393
  Loading UsnJrnl...
    UsnJrnl records: 245,425
  Preprocessing MFT reference table...
  Processing LogFile events...
  Processing UsnJrnl events...
  Normalizing event schemas...
  Combining and sorting events...

  --- Summary ---
  Total events: 375,137
  Unique files: 109,985
  Timestamp changes: 26,280
  Output: grouped_events_03-PE.csv (100.78 MB)


In [23]:
# [Cell 16] Process 04-PE
stats_04PE = process_dataset("04-PE", "PE")



PROCESSING: 04-PE (PE)
  Loading MFT...
    MFT records: 685,543
  Loading LogFile...
    LogFile records: 191,294
  Loading UsnJrnl...
    UsnJrnl records: 263,451
  Preprocessing MFT reference table...
  Processing LogFile events...
  Processing UsnJrnl events...
  Normalizing event schemas...
  Combining and sorting events...

  --- Summary ---
  Total events: 345,288
  Unique files: 7,882
  Timestamp changes: 11,936
  Output: grouped_events_04-PE.csv (81.74 MB)


In [24]:
# [Cell 17] Process 05-PE
stats_05PE = process_dataset("05-PE", "PE")



PROCESSING: 05-PE (PE)
  Loading MFT...
    MFT records: 685,545
  Loading LogFile...
    LogFile records: 200,385
  Loading UsnJrnl...
    UsnJrnl records: 265,287
  Preprocessing MFT reference table...
  Processing LogFile events...
  Processing UsnJrnl events...
  Normalizing event schemas...
  Combining and sorting events...

  --- Summary ---
  Total events: 349,098
  Unique files: 9,451
  Timestamp changes: 14,374
  Output: grouped_events_05-PE.csv (82.89 MB)


In [25]:
# [Cell 18] Process 06-PE
stats_06PE = process_dataset("06-PE", "PE")



PROCESSING: 06-PE (PE)
  Loading MFT...
    MFT records: 685,544
  Loading LogFile...
    LogFile records: 196,692
  Loading UsnJrnl...
    UsnJrnl records: 264,518
  Preprocessing MFT reference table...
  Processing LogFile events...
  Processing UsnJrnl events...
  Normalizing event schemas...
  Combining and sorting events...

  --- Summary ---
  Total events: 347,322
  Unique files: 9,434
  Timestamp changes: 14,517
  Output: grouped_events_06-PE.csv (82.26 MB)


In [26]:
# [Cell 19] Process 07-PE
stats_07PE = process_dataset("07-PE", "PE")



PROCESSING: 07-PE (PE)
  Loading MFT...
    MFT records: 685,483
  Loading LogFile...
    LogFile records: 320,155
  Loading UsnJrnl...
    UsnJrnl records: 247,908
  Preprocessing MFT reference table...
  Processing LogFile events...
  Processing UsnJrnl events...
  Normalizing event schemas...
  Combining and sorting events...

  --- Summary ---
  Total events: 384,439
  Unique files: 110,336
  Timestamp changes: 28,129
  Output: grouped_events_07-PE.csv (102.95 MB)


In [27]:
# [Cell 20] Process 08-PE
stats_08PE = process_dataset("08-PE", "PE")



PROCESSING: 08-PE (PE)
  Loading MFT...
    MFT records: 685,483
  Loading LogFile...
    LogFile records: 308,697
  Loading UsnJrnl...
    UsnJrnl records: 248,604
  Preprocessing MFT reference table...
  Processing LogFile events...
  Processing UsnJrnl events...
  Normalizing event schemas...
  Combining and sorting events...

  --- Summary ---
  Total events: 379,433
  Unique files: 110,541
  Timestamp changes: 25,446
  Output: grouped_events_08-PE.csv (101.52 MB)


In [28]:
# [Cell 21] Process 09-PE
stats_09PE = process_dataset("09-PE", "PE")



PROCESSING: 09-PE (PE)
  Loading MFT...
    MFT records: 685,495
  Loading LogFile...
    LogFile records: 333,100
  Loading UsnJrnl...
    UsnJrnl records: 249,559
  Preprocessing MFT reference table...
  Processing LogFile events...
  Processing UsnJrnl events...
  Normalizing event schemas...
  Combining and sorting events...

  --- Summary ---
  Total events: 390,346
  Unique files: 111,889
  Timestamp changes: 27,206
  Output: grouped_events_09-PE.csv (103.82 MB)


In [29]:
# [Cell 22] Process 10-PE
stats_10PE = process_dataset("10-PE", "PE")



PROCESSING: 10-PE (PE)
  Loading MFT...
    MFT records: 685,497
  Loading LogFile...
    LogFile records: 314,956
  Loading UsnJrnl...
    UsnJrnl records: 249,438
  Preprocessing MFT reference table...
  Processing LogFile events...
  Processing UsnJrnl events...
  Normalizing event schemas...
  Combining and sorting events...

  --- Summary ---
  Total events: 383,741
  Unique files: 110,574
  Timestamp changes: 25,388
  Output: grouped_events_10-PE.csv (102.15 MB)


In [30]:
# [Cell 23] Process 11-PE
stats_11PE = process_dataset("11-PE", "PE")



PROCESSING: 11-PE (PE)
  Loading MFT...
    MFT records: 685,544
  Loading LogFile...
    LogFile records: 196,099
  Loading UsnJrnl...
    UsnJrnl records: 264,432
  Preprocessing MFT reference table...
  Processing LogFile events...
  Processing UsnJrnl events...
  Normalizing event schemas...
  Combining and sorting events...

  --- Summary ---
  Total events: 347,441
  Unique files: 9,415
  Timestamp changes: 14,469
  Output: grouped_events_11-PE.csv (82.33 MB)


In [31]:
# [Cell 24] Process 12-PE
stats_12PE = process_dataset("12-PE", "PE")



PROCESSING: 12-PE (PE)
  Loading MFT...
    MFT records: 685,551
  Loading LogFile...
    LogFile records: 200,049
  Loading UsnJrnl...
    UsnJrnl records: 265,621
  Preprocessing MFT reference table...
  Processing LogFile events...
  Processing UsnJrnl events...
  Normalizing event schemas...
  Combining and sorting events...

  --- Summary ---
  Total events: 350,078
  Unique files: 9,160
  Timestamp changes: 13,529
  Output: grouped_events_12-PE.csv (82.59 MB)


## Training Datasets: APT (10 datasets)

Training APT datasets for model training.


In [32]:
# [Cell 26] Process 01-APT17
stats_01APT17 = process_dataset("01-APT17", "APT")



PROCESSING: 01-APT17 (APT)
  Loading MFT...
    MFT records: 374,033
  Loading LogFile...
    LogFile records: 402,654
  Loading UsnJrnl...
    UsnJrnl records: 319,041
  Preprocessing MFT reference table...
  Processing LogFile events...
  Processing UsnJrnl events...
  Normalizing event schemas...
  Combining and sorting events...

  --- Summary ---
  Total events: 490,994
  Unique files: 31,455
  Timestamp changes: 42,337
  Output: grouped_events_01-APT17.csv (116.37 MB)


In [33]:
# [Cell 27] Process 03-APT21
stats_03APT21 = process_dataset("03-APT21", "APT")



PROCESSING: 03-APT21 (APT)
  Loading MFT...
    MFT records: 374,039
  Loading LogFile...
    LogFile records: 397,743
  Loading UsnJrnl...
    UsnJrnl records: 317,324
  Preprocessing MFT reference table...
  Processing LogFile events...
  Processing UsnJrnl events...
  Normalizing event schemas...
  Combining and sorting events...

  --- Summary ---
  Total events: 487,612
  Unique files: 28,028
  Timestamp changes: 36,871
  Output: grouped_events_03-APT21.csv (114.49 MB)


In [34]:
# [Cell 28] Process 04-APT28
stats_04APT28 = process_dataset("04-APT28", "APT")



PROCESSING: 04-APT28 (APT)
  Loading MFT...
    MFT records: 374,018
  Loading LogFile...
    LogFile records: 406,714
  Loading UsnJrnl...
    UsnJrnl records: 322,260
  Preprocessing MFT reference table...
  Processing LogFile events...
  Processing UsnJrnl events...
  Normalizing event schemas...
  Combining and sorting events...

  --- Summary ---
  Total events: 494,522
  Unique files: 29,399
  Timestamp changes: 38,444
  Output: grouped_events_04-APT28.csv (116.57 MB)


In [35]:
# [Cell 29] Process 05-APT29
stats_05APT29 = process_dataset("05-APT29", "APT")



PROCESSING: 05-APT29 (APT)
  Loading MFT...
    MFT records: 374,017
  Loading LogFile...
    LogFile records: 406,491
  Loading UsnJrnl...
    UsnJrnl records: 328,212
  Preprocessing MFT reference table...
  Processing LogFile events...
  Processing UsnJrnl events...
  Normalizing event schemas...
  Combining and sorting events...

  --- Summary ---
  Total events: 502,832
  Unique files: 29,860
  Timestamp changes: 37,626
  Output: grouped_events_05-APT29.csv (117.27 MB)


In [36]:
# [Cell 30] Process 06-APT30
stats_06APT30 = process_dataset("06-APT30", "APT")



PROCESSING: 06-APT30 (APT)
  Loading MFT...
    MFT records: 374,015
  Loading LogFile...
    LogFile records: 391,417
  Loading UsnJrnl...
    UsnJrnl records: 328,772
  Preprocessing MFT reference table...
  Processing LogFile events...
  Processing UsnJrnl events...
  Normalizing event schemas...
  Combining and sorting events...

  --- Summary ---
  Total events: 499,102
  Unique files: 22,266
  Timestamp changes: 27,896
  Output: grouped_events_06-APT30.csv (116.23 MB)


In [37]:
# [Cell 31] Process 07-APT37
stats_07APT37 = process_dataset("07-APT37", "APT")



PROCESSING: 07-APT37 (APT)
  Loading MFT...
    MFT records: 374,018
  Loading LogFile...
    LogFile records: 401,924
  Loading UsnJrnl...
    UsnJrnl records: 319,578
  Preprocessing MFT reference table...
  Processing LogFile events...
  Processing UsnJrnl events...
  Normalizing event schemas...
  Combining and sorting events...

  --- Summary ---
  Total events: 491,554
  Unique files: 29,426
  Timestamp changes: 38,646
  Output: grouped_events_07-APT37.csv (114.68 MB)


In [38]:
# [Cell 32] Process 08-APT38
stats_08APT38 = process_dataset("08-APT38", "APT")



PROCESSING: 08-APT38 (APT)
  Loading MFT...
    MFT records: 374,022
  Loading LogFile...
    LogFile records: 384,306
  Loading UsnJrnl...
    UsnJrnl records: 317,216
  Preprocessing MFT reference table...
  Processing LogFile events...
  Processing UsnJrnl events...
  Normalizing event schemas...
  Combining and sorting events...

  --- Summary ---
  Total events: 483,190
  Unique files: 20,478
  Timestamp changes: 28,471
  Output: grouped_events_08-APT38.csv (112.91 MB)


In [39]:
# [Cell 33] Process 10-DarkHotel663
stats_10DarkHotel663 = process_dataset("10-DarkHotel663", "APT")



PROCESSING: 10-DarkHotel663 (APT)
  Loading MFT...
    MFT records: 374,014
  Loading LogFile...
    LogFile records: 331,594
  Loading UsnJrnl...
    UsnJrnl records: 305,678
  Preprocessing MFT reference table...
  Processing LogFile events...
  Processing UsnJrnl events...
  Normalizing event schemas...
  Combining and sorting events...

  --- Summary ---
  Total events: 453,064
  Unique files: 10,625
  Timestamp changes: 18,396
  Output: grouped_events_10-DarkHotel663.csv (115.00 MB)


In [40]:
# [Cell 34] Process 11-DarkHotelbbd
stats_11DarkHotelbbd = process_dataset("11-DarkHotelbbd", "APT")



PROCESSING: 11-DarkHotelbbd (APT)
  Loading MFT...
    MFT records: 374,012
  Loading LogFile...
    LogFile records: 331,216
  Loading UsnJrnl...
    UsnJrnl records: 305,506
  Preprocessing MFT reference table...
  Processing LogFile events...
  Processing UsnJrnl events...
  Normalizing event schemas...
  Combining and sorting events...

  --- Summary ---
  Total events: 452,735
  Unique files: 10,536
  Timestamp changes: 18,336
  Output: grouped_events_11-DarkHotelbbd.csv (111.05 MB)


In [41]:
# [Cell 35] Process 14-Winnti43b
stats_14Winnti43b = process_dataset("14-Winnti53b", "APT")



PROCESSING: 14-Winnti53b (APT)
  Loading MFT...
    MFT records: 374,026
  Loading LogFile...
    LogFile records: 391,419
  Loading UsnJrnl...
    UsnJrnl records: 248,403
  Preprocessing MFT reference table...
  Processing LogFile events...
  Processing UsnJrnl events...
  Normalizing event schemas...
  Combining and sorting events...

  --- Summary ---
  Total events: 424,926
  Unique files: 16,703
  Timestamp changes: 28,070
  Output: grouped_events_14-Winnti53b.csv (103.91 MB)


## Validation Datasets (5 datasets)

These datasets are processed independently and will NOT be included in the training merge.
Each will be used for independent model evaluation.


In [42]:
# [Cell 37] Process 02-APT19 (Validation)
stats_02APT19 = process_dataset("02-APT19", "APT")



PROCESSING: 02-APT19 (APT)
  Loading MFT...
    MFT records: 374,034
  Loading LogFile...
    LogFile records: 346,222
  Loading UsnJrnl...
    UsnJrnl records: 325,744
  Preprocessing MFT reference table...
  Processing LogFile events...
  Processing UsnJrnl events...
  Normalizing event schemas...
  Combining and sorting events...

  --- Summary ---
  Total events: 475,043
  Unique files: 21,336
  Timestamp changes: 26,358
  Output: grouped_events_02-APT19.csv (112.24 MB)


In [43]:
# [Cell 38] Process 09-APT40 (Validation)
stats_09APT40 = process_dataset("09-APT40", "APT")



PROCESSING: 09-APT40 (APT)
  Loading MFT...
    MFT records: 374,018
  Loading LogFile...
    LogFile records: 405,109
  Loading UsnJrnl...
    UsnJrnl records: 323,226
  Preprocessing MFT reference table...
  Processing LogFile events...
  Processing UsnJrnl events...
  Normalizing event schemas...
  Combining and sorting events...

  --- Summary ---
  Total events: 497,444
  Unique files: 21,672
  Timestamp changes: 29,817
  Output: grouped_events_09-APT40.csv (116.29 MB)


In [44]:
# [Cell 39] Process 12-Kimsuky (Validation)
stats_12Kimsuky = process_dataset("12-Kimsuky", "APT")



PROCESSING: 12-Kimsuky (APT)
  Loading MFT...
    MFT records: 374,015
  Loading LogFile...
    LogFile records: 341,101
  Loading UsnJrnl...
    UsnJrnl records: 305,554
  Preprocessing MFT reference table...
  Processing LogFile events...
  Processing UsnJrnl events...
  Normalizing event schemas...
  Combining and sorting events...

  --- Summary ---
  Total events: 454,971
  Unique files: 14,869
  Timestamp changes: 22,752
  Output: grouped_events_12-Kimsuky.csv (110.54 MB)


In [45]:
# [Cell 40] Process 13-Winnti731 (Validation)
stats_13Winnti731 = process_dataset("13-Winnti731", "APT")



PROCESSING: 13-Winnti731 (APT)
  Loading MFT...
    MFT records: 374,012
  Loading LogFile...
    LogFile records: 354,002
  Loading UsnJrnl...
    UsnJrnl records: 245,424
  Preprocessing MFT reference table...
  Processing LogFile events...
  Processing UsnJrnl events...
  Normalizing event schemas...
  Combining and sorting events...

  --- Summary ---
  Total events: 401,719
  Unique files: 16,741
  Timestamp changes: 27,555
  Output: grouped_events_13-Winnti731.csv (97.84 MB)


In [46]:
# [Cell 41] Process LoneWolf (Validation)
stats_LoneWolf = process_dataset("LoneWolf", "LoneWolf")



PROCESSING: LoneWolf (LoneWolf)
  Loading MFT...
    MFT records: 142,960
  Loading LogFile...
    LogFile records: 294,761
  Loading UsnJrnl...
    UsnJrnl records: 352,849
  Preprocessing MFT reference table...
  Processing LogFile events...
  Processing UsnJrnl events...
  Normalizing event schemas...
  Combining and sorting events...

  --- Summary ---
  Total events: 481,400
  Unique files: 17,897
  Timestamp changes: 20,373
  Output: grouped_events_LoneWolf.csv (114.55 MB)


## Merge Training Datasets

Combine all 22 training dataset CSVs into a single `grouped_events_training.csv`.
This will be used for Phase 3 feature engineering and Phase 4 model training.


In [47]:
# [Cell 43] Merge Training Datasets

print("=" * 60)
print("MERGING TRAINING DATASETS")
print("=" * 60)

# List of all training datasets
training_datasets = TRAINING_PE + TRAINING_APT
print(f"\nTraining datasets to merge: {len(training_datasets)}")

# Load and concatenate
training_dfs = []
for dataset in training_datasets:
    csv_path = OUTPUT_DIR / f"grouped_events_{dataset}.csv"
    if csv_path.exists():
        print(f"  Loading {dataset}...")
        df = pd.read_csv(csv_path, low_memory=False)
        training_dfs.append(df)
        print(f"    Records: {len(df):,}")
    else:
        print(f"  WARNING: {csv_path.name} not found, skipping...")

if training_dfs:
    # Concatenate all training data
    print("\n  Concatenating all training datasets...")
    df_training = pd.concat(training_dfs, ignore_index=True)
    
    # Export combined training CSV
    training_output = OUTPUT_DIR / "grouped_events_training.csv"
    df_training.to_csv(training_output, index=False, encoding='utf-8')
    training_size_mb = training_output.stat().st_size / (1024 * 1024)
    
    print(f"\n--- Training Data Merged ---")
    print(f"Total records: {len(df_training):,}")
    print(f"Unique files: {df_training['FileFRN'].nunique():,}")
    print(f"Datasets included: {df_training['dataID'].nunique()}")
    print(f"Output: {training_output.name} ({training_size_mb:.2f} MB)")
else:
    print("\nERROR: No training datasets found to merge.")


MERGING TRAINING DATASETS

Training datasets to merge: 22
  Loading 01-PE...
    Records: 505,582
  Loading 02-PE...
    Records: 363,619
  Loading 03-PE...
    Records: 375,137
  Loading 04-PE...
    Records: 345,288
  Loading 05-PE...
    Records: 349,098
  Loading 06-PE...
    Records: 347,322
  Loading 07-PE...
    Records: 384,439
  Loading 08-PE...
    Records: 379,433
  Loading 09-PE...
    Records: 390,346
  Loading 10-PE...
    Records: 383,741
  Loading 11-PE...
    Records: 347,441
  Loading 12-PE...
    Records: 350,078
  Loading 01-APT17...
    Records: 490,994
  Loading 03-APT21...
    Records: 487,612
  Loading 04-APT28...
    Records: 494,522
  Loading 05-APT29...
    Records: 502,832
  Loading 06-APT30...
    Records: 499,102
  Loading 07-APT37...
    Records: 491,554
  Loading 08-APT38...
    Records: 483,190
  Loading 10-DarkHotel663...
    Records: 453,064
  Loading 11-DarkHotelbbd...
    Records: 452,735

  Concatenating all training datasets...

--- Training Data 

## Processing Summary

Collect and display statistics for all processed datasets.


In [48]:
# [Cell 45] Generate Summary Report

print("=" * 60)
print("PHASE 2 PROCESSING SUMMARY")
print("=" * 60)

# Collect all stats variables
all_stats = []
for var_name in dir():
    if var_name.startswith('stats_') and isinstance(eval(var_name), dict):
        all_stats.append(eval(var_name))

if all_stats:
    # Create summary DataFrame
    df_summary = pd.DataFrame(all_stats)
    
    # Display summary
    print(f"\nDatasets processed: {len(df_summary)}")
    print(f"\n{'Dataset':<20} {'Events':>12} {'Files':>10} {'TS Changes':>12} {'Size (MB)':>10}")
    print("-" * 70)
    
    for _, row in df_summary.iterrows():
        print(f"{row['dataset']:<20} {row['total_events']:>12,} {row['unique_files']:>10,} {row['timestamp_changes']:>12,} {row['output_size_mb']:>10.2f}")
    
    print("-" * 70)
    print(f"{'TOTAL':<20} {df_summary['total_events'].sum():>12,} {df_summary['unique_files'].sum():>10,} {df_summary['timestamp_changes'].sum():>12,} {df_summary['output_size_mb'].sum():>10.2f}")
    
    # Save summary
    summary_path = OUTPUT_DIR / "phase2_processing_summary.csv"
    df_summary.to_csv(summary_path, index=False)
    print(f"\nSummary saved to: {summary_path}")
else:
    print("\nNo processing statistics available. Run dataset cells first.")

print("\n" + "=" * 60)
print("PHASE 2 COMPLETE")
print("=" * 60)
print("\nNext Step: Phase 3 - Feature Engineering & Labeling")


PHASE 2 PROCESSING SUMMARY

Datasets processed: 27

Dataset                    Events      Files   TS Changes  Size (MB)
----------------------------------------------------------------------
01-APT17                  490,994     31,455       42,337     116.37
01-PE                     505,582     37,375       32,435     122.67
02-APT19                  475,043     21,336       26,358     112.24
02-PE                     363,619    105,251       14,978      95.41
03-APT21                  487,612     28,028       36,871     114.49
03-PE                     375,137    109,985       26,280     100.78
04-APT28                  494,522     29,399       38,444     116.57
04-PE                     345,288      7,882       11,936      81.74
05-APT29                  502,832     29,860       37,626     117.27
05-PE                     349,098      9,451       14,374      82.89
06-APT30                  499,102     22,266       27,896     116.23
06-PE                     347,322      9,434     

## Output File Reference

### Training Data (Combined)
- `grouped_events_training.csv` - All 22 training datasets merged

### Training Datasets (Individual)
- `grouped_events_01-PE.csv` through `grouped_events_12-PE.csv`
- `grouped_events_01-APT17.csv`, `grouped_events_03-APT21.csv`, etc.

### Validation Datasets (Separate)
- `grouped_events_02-APT19.csv`
- `grouped_events_09-APT40.csv`
- `grouped_events_12-Kimsuky.csv`
- `grouped_events_13-Winnti731.csv`
- `grouped_events_LoneWolf.csv`

### Processing Summary
- `phase2_processing_summary.csv` - Statistics for all datasets